# Notebook 01 — Dataset Preprocessing

**Pipeline:** Protein-Aware Multimodal Multi-Task QSAR — Endocrine Toxicity  
**Paper:** *Protein-Conditioned Multimodal Multi-Task QSAR Classification of IC50-Based Endocrine Toxicity Mechanism Endpoints across AR, CYP19A1, ESR1, ESR2, HSD11B1, and PPARG under Stratified Scaffold Evaluation*

---

## What this notebook does

This notebook covers the full data preprocessing pipeline from raw ChEMBL bioactivity exports to clean, split-ready datasets:

| Step | Description | Key output |
|------|-------------|------------|
| 1 | Load & validate raw ChEMBL CSVs | Standardized DataFrames per target |
| 2 | Molecular standardization | Canonical SMILES, largest fragment |
| 3 | Unit conversion & pIC50 | IC50 in nM, pIC50 column |
| 4 | Clean-margin binary labeling | Labels: 1 (active) / 0 (non-active) / removed |
| 5 | Deduplication | Median IC50 per molecule per endpoint |
| 6 | Bemis-Murcko scaffold assignment | Scaffold column for split strategy |
| 7 | Class balancing | Max 3:1 majority:minority via undersampling |
| 8 | Stratified scaffold split (80/20) | `split` column: train_pool / heldout_test |
| 9 | Leakage check | Zero scaffold & molecule overlap confirmed |
| 10 | Save & summary statistics | Per-endpoint train/test files + Table 1 |

---

## Prerequisites

Before running this notebook:
1. Download the 6 ChEMBL IC50 CSV files — see `data/raw/README.md` for instructions.
2. Set the correct paths in `config.yaml` (`raw_data_root` and `processed_root`).
3. Install dependencies: `pip install -r requirements.txt`

## Runtime

| Mode | Approx. time | GPU needed? |
|------|-------------|-------------|
| **Full** (all 6 targets, ~13k molecules) | ~5–10 min | No |
| **Demo** (200 molecules per target) | ~1 min | No |

---
## 0 — Imports & Configuration

All configuration is loaded from `config.yaml` at the root of the repo.  
**The only thing you need to change is the `paths` section in that file.**

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import yaml
warnings.filterwarnings('ignore')

# ── Add repo root to path so we can import from utils/ ───────────────────────
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.chem_utils import (
    get_largest_fragment,
    get_canonical_smiles,
    get_bemis_murcko_scaffold,
    convert_to_nM,
    compute_pIC50,
    normalize_relation,
    assign_binary_label,
    assign_removed_reason,
)
from utils.split_utils import (
    stratified_scaffold_split,
    make_cv_fold_assignments,
    check_leakage,
    summarize_split,
)

print('✅ Imports OK')

In [ ]:
# ── Load config ───────────────────────────────────────────────────────────────
with open(REPO_ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

SEED         = CFG['seed']
DEMO_MODE    = CFG['demo']['enabled']
DEMO_N       = CFG['demo']['n_mols']

# Thresholds (locked to paper methodology — do not change)
ACTIVE_THR   = CFG['labeling']['active_threshold_nM']    # 1000 nM
INACTIVE_THR = CFG['labeling']['inactive_threshold_nM']  # 10000 nM
MAX_RATIO    = CFG['balancing']['max_ratio']              # 3:1
TEST_SIZE    = CFG['split']['test_size']                  # 0.20
N_CV_FOLDS   = CFG['split']['n_cv_folds']                 # 10

# ── Paths ─────────────────────────────────────────────────────────────────────
# These come from config.yaml → paths section.
# The dummy paths below will be replaced automatically once you set
# the correct paths in config.yaml.
RAW_ROOT       = Path(CFG['paths']['raw_data_root'])
PROCESSED_ROOT = Path(CFG['paths']['processed_root'])
SPLIT_ROOT     = Path(CFG['paths']['split_root'])

SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

# ── Target list ───────────────────────────────────────────────────────────────
TARGETS = CFG['targets']  # list of dicts with keys: name, chembl_id, label

print(f'✅ Config loaded')
print(f'   DEMO_MODE    : {DEMO_MODE}')
print(f'   SEED         : {SEED}')
print(f'   Active thr   : IC50 <= {ACTIVE_THR} nM')
print(f'   Inactive thr : IC50 >= {INACTIVE_THR} nM')
print(f'   Max ratio    : {MAX_RATIO}:1')
print(f'   Test size    : {TEST_SIZE}')
print(f'   N CV folds   : {N_CV_FOLDS}')
print(f'   Targets      : {[t["name"] for t in TARGETS]}')
print(f'   RAW_ROOT     : {RAW_ROOT}')
print(f'   SPLIT_ROOT   : {SPLIT_ROOT}')

if DEMO_MODE:
    print(f'\n  ⚡ DEMO MODE ON — using {DEMO_N} molecules per target')

---
## 1 — Load Raw ChEMBL CSVs

ChEMBL exports can use either `,` or `;` as delimiter depending on regional settings.  
This cell auto-detects the delimiter and normalizes column names.

> **Expected file location:**  
> `{raw_data_root}/{chembl_id} ({target_name}) - IC50.csv`  
> See `data/raw/README.md` for download instructions.

In [ ]:
def load_chembl_csv(filepath: Path) -> pd.DataFrame:
    """
    Load a ChEMBL bioactivity CSV with automatic delimiter detection.
    Returns a DataFrame with cleaned column names.
    """
    # Try comma first, then semicolon
    for sep in [',', ';']:
        try:
            df = pd.read_csv(
                filepath,
                sep=sep,
                dtype=str,
                encoding='utf-8-sig',
            )
            # A successful parse has multiple real columns (not one giant string)
            if df.shape[1] > 5:
                # Clean column names: strip whitespace, quotes, and trailing commas
                df.columns = (
                    df.columns
                    .str.strip()
                    .str.strip('"')
                    .str.strip("'")
                    .str.split(',').str[0]   # handle 'ColName,...' artifacts
                )
                # Drop unnamed columns
                df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
                return df
        except Exception:
            continue

    raise ValueError(f'Could not parse {filepath} with delimiter , or ;')


# ── Map target name → raw file path ──────────────────────────────────────────
# Adjust the filename pattern if your downloaded files use different naming.
# Format: '{chembl_id} ({target_label}) - IC50.csv'
TARGET_FILE_MAP = {}
for t in TARGETS:
    # Try to find the file automatically
    pattern_candidates = [
        RAW_ROOT / t['name'] / f"{t['chembl_id']} ({t['label']}) - IC50.csv",
        RAW_ROOT / f"{t['chembl_id']} ({t['label']}) - IC50.csv",
    ]
    found = next((p for p in pattern_candidates if p.exists()), None)
    TARGET_FILE_MAP[t['name']] = found


# ── Load all raw data ─────────────────────────────────────────────────────────
raw_dfs = {}
for target_name, filepath in TARGET_FILE_MAP.items():
    if filepath is None or not filepath.exists():
        print(f'  ⚠️  [{target_name}] File not found — skipping.')
        print(f'     Expected path: see config.yaml → paths.raw_data_root')
        continue

    df = load_chembl_csv(filepath)
    df['_source_target'] = target_name  # tag origin
    raw_dfs[target_name] = df

    print(f'  ✅ [{target_name}] Loaded {df.shape[0]:,} records | {df.shape[1]} columns')

if not raw_dfs:
    print('\n  No raw files found. Running in DEMO MODE with synthetic data.')
    # DEMO_MODE will handle this in the next cell

print(f'\nTotal targets loaded: {len(raw_dfs)}')

---
## 1b — Demo Mode: Generate Toy Dataset *(skip if using real data)*

If `demo.enabled: true` in `config.yaml`, or if no raw files were found,  
this cell creates a small synthetic dataset from a curated set of known molecules  
so you can run the full pipeline end-to-end without downloading data.

In [ ]:
if DEMO_MODE or not raw_dfs:
    from rdkit import Chem
    from rdkit.Chem import RDConfig
    import random
    random.seed(SEED)
    np.random.seed(SEED)

    print('⚡ Generating demo dataset...')

    # Small set of diverse drug-like SMILES (publicly known compounds)
    BASE_SMILES = [
        'CC1=CC=C(C=C1)S(=O)(=O)NC2=CC=C(N)C=C2',    # dapsone
        'CC(=O)Nc1ccc(O)cc1',                          # acetaminophen
        'c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34',            # pyrene
        'O=C(O)c1ccccc1O',                             # salicylic acid
        'CCOc1ccc(NC(=O)c2ccncc2)cc1',                 # analog
        'O=C(NCc1ccccc1)c1ccncc1',                     # analog
        'Cc1ccc(NC(=O)Nc2ccc(Cl)c(C(F)(F)F)c2)cc1',   # sorafenib analog
        'CC(C)Cc1ccc(C(C)C(=O)O)cc1',                  # ibuprofen
        'O=C(O)c1cc(Cl)ccc1Nc1ccnc2cc(Cl)ccc12',       # analog
        'Nc1ccc(C(=O)O)cc1',                           # aminobenzoic acid
        'CC1=C(C(=O)Nc2ccccc2)C(c2ccccc2Cl)NC(=O)C1',  # nifedipine analog
        'O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O',  # adenosine
        'Cc1cc(=O)[nH]c(=O)[nH]1',                     # thymine
        'NC(=O)c1ccc[n+]([C@@H]2O[C@H](COP(=O)(O)O)[C@@H](O)[C@H]2O)c1',
        'CC(=O)OC1=CC=CC=C1C(=O)O',                    # aspirin
    ]

    demo_rows = []
    for target in TARGETS:
        rng_local = np.random.RandomState(SEED + hash(target['name']) % 1000)
        smiles_pool = BASE_SMILES * (DEMO_N // len(BASE_SMILES) + 2)
        selected    = smiles_pool[:DEMO_N]

        for i, smi in enumerate(selected):
            # Assign random IC50 values spanning all three zones
            ic50 = rng_local.choice(
                [rng_local.uniform(100, 999),     # active zone
                 rng_local.uniform(1001, 9999),   # intermediate (will be removed)
                 rng_local.uniform(10001, 50000)] # inactive zone
            )
            demo_rows.append({
                'Smiles':              smi,
                'Standard Type':       'IC50',
                'Standard Relation':   '=',
                'Standard Value':      str(ic50),
                'Standard Units':      'nM',
                'pChEMBL Value':       '',
                'Data Validity Comment': '',
                'Assay Confidence Score': '9',
                'Molecule ChEMBL ID':  f'DEMO_{target["name"]}_{i:04d}',
                '_source_target':      target['name'],
            })

    raw_dfs = defaultdict(pd.DataFrame)
    for row in demo_rows:
        t = row['_source_target']
        raw_dfs[t] = pd.concat(
            [raw_dfs[t], pd.DataFrame([row])], ignore_index=True
        )

    for target_name, df in raw_dfs.items():
        print(f'  ✅ [DEMO] [{target_name}] {len(df)} rows')

else:
    print('Real data loaded — skipping demo generation.')

---
## 2 — Molecular Standardization

For each record:
1. Parse SMILES with RDKit
2. For salts/mixtures: retain largest organic fragment
3. Generate canonical SMILES (unified representation)
4. Drop records with invalid SMILES

In [ ]:
standardized_dfs = {}

for target_name, df in raw_dfs.items():
    n_raw = len(df)

    # ── Step 1: Require a non-null SMILES column ──────────────────────────────
    df = df.dropna(subset=['Smiles']).copy()
    df['Smiles'] = df['Smiles'].astype(str).str.strip()

    # ── Step 2: Largest fragment (handles salts like "CC(=O)[O-].[Na+]") ─────
    df['canonical_smiles'] = df['Smiles'].apply(get_largest_fragment)

    # ── Step 3: Drop molecules that failed RDKit parsing ─────────────────────
    n_before = len(df)
    df = df.dropna(subset=['canonical_smiles'])
    n_invalid = n_before - len(df)

    standardized_dfs[target_name] = df

    print(
        f'  [{target_name}] '
        f'Raw: {n_raw:,} | '
        f'After SMILES filter: {len(df):,} '
        f'(dropped {n_raw - len(df):,} missing/invalid)'
    )

print('\n✅ Molecular standardization complete.')

---
## 3 — Unit Conversion, pIC50, and Quality Filters

- Convert all IC50 values to nanomolar (nM)
- Compute pIC50 = 9 − log₁₀(IC50_nM)
- Apply quality filters:
  - Standard Relation must be `=` (exact measurement, not `>` or `<`)
  - Standard Value must be numeric and positive
  - Standard Units must be a recognized unit (nM, uM, µM, mM, M, pM)

In [ ]:
ALLOWED_RELATIONS = CFG['filters']['allowed_relations']  # ['=']

filtered_dfs = {}

for target_name, df in standardized_dfs.items():
    df = df.copy()

    # ── Standard Value: coerce to numeric ─────────────────────────────────────
    df['Standard Value Numeric'] = pd.to_numeric(
        df['Standard Value'], errors='coerce'
    )

    # ── Normalize Standard Relation ───────────────────────────────────────────
    df['Standard Relation Clean'] = df['Standard Relation'].apply(normalize_relation)

    # ── Convert to nM ─────────────────────────────────────────────────────────
    df['IC50_nM'] = df.apply(
        lambda r: convert_to_nM(r['Standard Value Numeric'], r['Standard Units']),
        axis=1,
    )

    # ── Compute pIC50 ─────────────────────────────────────────────────────────
    df['pIC50'] = df['IC50_nM'].apply(compute_pIC50)

    # ── Quality filters ───────────────────────────────────────────────────────
    n_before = len(df)
    df = df[
        df['Standard Relation Clean'].isin(ALLOWED_RELATIONS) &
        df['IC50_nM'].notna() &
        (df['IC50_nM'] > 0)
    ].copy()
    n_filtered = n_before - len(df)

    filtered_dfs[target_name] = df

    print(
        f'  [{target_name}] '
        f'After quality filter: {len(df):,} '
        f'(removed {n_filtered:,} — non-exact relation or invalid value)'
    )

print('\n✅ Unit conversion and quality filtering complete.')

---
## 4 — Clean-Margin Binary Labeling

Labels are assigned using a **clean-margin threshold** strategy to minimize label noise:

| IC50 (nM) | Label | Biological meaning |
|-----------|-------|--------------------|
| ≤ 1,000   | **1** (active) | Strong target activity — relevant EDC activity |
| 1,000–10,000 | **removed** | Intermediate zone — ambiguous, high label noise |
| ≥ 10,000  | **0** (non-active) | No meaningful activity at this threshold |

> ⚠️ Labels represent **molecular-level target activity**, not clinical toxicity directly.

In [ ]:
labeled_dfs  = {}   # records that received label 0 or 1
removed_dfs  = {}   # intermediate zone records (for audit)

for target_name, df in filtered_dfs.items():
    df = df.copy()

    df['label'] = df['IC50_nM'].apply(
        lambda x: assign_binary_label(x, ACTIVE_THR, INACTIVE_THR)
    )
    df['label_name'] = df['label'].map({1: 'Active', 0: 'Non-active'})

    # Split into labeled vs removed
    df_labeled  = df[df['label'].isin([0, 1])].copy()
    df_removed  = df[~df['label'].isin([0, 1])].copy()
    df_removed['removed_reason'] = df_removed.apply(assign_removed_reason, axis=1)

    labeled_dfs[target_name] = df_labeled
    removed_dfs[target_name] = df_removed

    n_act = (df_labeled['label'] == 1).sum()
    n_ina = (df_labeled['label'] == 0).sum()
    n_rem = len(df_removed)

    print(
        f'  [{target_name}] '
        f'Active: {n_act:,} | '
        f'Non-active: {n_ina:,} | '
        f'Removed (intermediate): {n_rem:,} | '
        f'Ratio: {n_act/max(n_ina,1):.2f}:1'
    )

print('\n✅ Binary labeling complete.')

---
## 5 — Deduplication

Multiple assay records can exist for the same molecule in ChEMBL  
(different assays, different labs, different years).

**Strategy:** For each (canonical SMILES, endpoint) pair:
- Compute the **median IC50** across all duplicates.
- If the median falls in the intermediate zone (1,000–10,000 nM) → remove.
- If individual records span both active and non-active zones  
  (irreconcilable inter-assay disagreement) → remove.
- Otherwise, assign the median-based label.

In [ ]:
deduped_dfs = {}

for target_name, df in labeled_dfs.items():
    n_before = len(df)

    # Group by canonical SMILES and aggregate
    dedup_rows = []
    for smi, grp in df.groupby('canonical_smiles'):
        ic50_values  = grp['IC50_nM'].dropna().values
        if len(ic50_values) == 0:
            continue

        median_ic50 = float(np.median(ic50_values))
        label       = assign_binary_label(median_ic50, ACTIVE_THR, INACTIVE_THR)

        # Check for irreconcilable disagreement:
        # individual records span both active (≤1000) and inactive (≥10000)
        has_active   = (ic50_values <= ACTIVE_THR).any()
        has_inactive = (ic50_values >= INACTIVE_THR).any()
        if has_active and has_inactive and label is None:
            # Conflicting — exclude
            continue

        if label is None:
            # Median fell in intermediate zone after dedup
            continue

        # Keep one representative row (first occurrence) with updated IC50/pIC50/label
        best_row = grp.iloc[0].copy()
        best_row['IC50_nM']    = median_ic50
        best_row['pIC50']      = compute_pIC50(median_ic50)
        best_row['label']      = label
        best_row['label_name'] = 'Active' if label == 1 else 'Non-active'
        dedup_rows.append(best_row)

    deduped_df = pd.DataFrame(dedup_rows).reset_index(drop=True)
    deduped_dfs[target_name] = deduped_df

    n_act = (deduped_df['label'] == 1).sum()
    n_ina = (deduped_df['label'] == 0).sum()

    print(
        f'  [{target_name}] '
        f'Before: {n_before:,} → After dedup: {len(deduped_df):,} unique molecules | '
        f'Active: {n_act:,} | Non-active: {n_ina:,}'
    )

print('\n✅ Deduplication complete.')

---
## 6 — Bemis-Murcko Scaffold Assignment

Scaffolds are used as the **grouping unit** for the stratified scaffold split.  
Compounds sharing the same scaffold will always be assigned to the same partition.

**Acyclic molecules** produce an empty scaffold string. These are assigned their  
canonical SMILES as a unique scaffold key (so they don't all collapse into one group).

In [ ]:
scaffold_dfs = {}

for target_name, df in deduped_dfs.items():
    df = df.copy()
    df['scaffold'] = df['canonical_smiles'].apply(get_bemis_murcko_scaffold)

    # Drop the few molecules for which scaffold generation failed
    n_fail = df['scaffold'].isna().sum()
    df = df.dropna(subset=['scaffold'])

    n_unique_scaffolds = df['scaffold'].nunique()
    n_unique_mols      = df['canonical_smiles'].nunique()

    scaffold_dfs[target_name] = df

    print(
        f'  [{target_name}] '
        f'Molecules: {n_unique_mols:,} | '
        f'Unique scaffolds: {n_unique_scaffolds:,} | '
        f'Scaffold:mol ratio: {n_unique_scaffolds/max(n_unique_mols,1):.2f}'
        + (f' | Failed: {n_fail}' if n_fail > 0 else '')
    )

print('\n✅ Scaffold assignment complete.')

---
## 7 — Class Balancing

Class imbalance is controlled by **random undersampling** of the majority class.  
Maximum allowed majority:minority ratio = **3:1** (from `config.yaml`).

> ⚠️ Balancing is applied **only in the train pool**, NOT in the held-out test set.  
> The held-out test retains the natural class distribution for realistic evaluation.

In [ ]:
def undersample_majority(
    df: pd.DataFrame,
    label_col: str = 'label',
    max_ratio: float = 3.0,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Undersample the majority class so that majority:minority <= max_ratio.
    All minority samples are kept. Returns the balanced DataFrame.
    """
    counts = df[label_col].value_counts()
    majority_label  = counts.idxmax()
    minority_label  = counts.idxmin()
    n_minority      = counts[minority_label]
    n_majority_max  = int(n_minority * max_ratio)

    if counts[majority_label] <= n_majority_max:
        return df  # already within ratio

    majority_df = df[df[label_col] == majority_label].sample(
        n=n_majority_max, random_state=seed, replace=False
    )
    minority_df = df[df[label_col] == minority_label]

    return pd.concat([majority_df, minority_df], ignore_index=True)


# NOTE: Balancing is applied AFTER splitting, but we do it here at the
# dataset level first to establish the molecule pool. Final balancing
# will be re-applied to the train pool only after split (Step 8).

balanced_dfs = {}

for target_name, df in scaffold_dfs.items():
    counts_before = df['label'].value_counts()
    ratio_before  = counts_before.max() / counts_before.min()

    df_balanced   = undersample_majority(df, max_ratio=MAX_RATIO, seed=SEED)
    counts_after  = df_balanced['label'].value_counts()
    ratio_after   = counts_after.max() / counts_after.min()

    balanced_dfs[target_name] = df_balanced

    print(
        f'  [{target_name}] '
        f'Before: {len(df):,} (ratio {ratio_before:.2f}:1) → '
        f'After: {len(df_balanced):,} (ratio {ratio_after:.2f}:1)'
    )

print('\n✅ Class balancing complete.')

---
## 8 — Combine & Stratified Scaffold Split

All 6 endpoint datasets are **combined into a single multi-endpoint matrix**  
before splitting. This ensures every molecule gets **one consistent** train/test  
assignment across all endpoints.

Split strategy:
- 80% scaffold groups → **train pool** (used for CV + model training)
- 20% scaffold groups → **held-out test** (used ONCE for final evaluation)
- Scaffold groups are kept intact — no scaffold appears in both partitions

In [ ]:
# ── Tag each record with its endpoint name ────────────────────────────────────
tagged = []
for target_name, df in balanced_dfs.items():
    df = df.copy()
    df['endpoint_name'] = target_name
    tagged.append(df)

combined = pd.concat(tagged, ignore_index=True)

print(f'Combined dataset: {len(combined):,} records across {combined["endpoint_name"].nunique()} endpoints')
print(f'Unique canonical SMILES: {combined["canonical_smiles"].nunique():,}')
print(f'Unique scaffolds: {combined["scaffold"].nunique():,}')

In [ ]:
# ── Perform stratified scaffold split ─────────────────────────────────────────
combined = stratified_scaffold_split(
    df           = combined,
    scaffold_col = 'scaffold',
    smiles_col   = 'canonical_smiles',
    endpoint_col = 'endpoint_name',
    label_col    = 'label',
    test_size    = TEST_SIZE,
    seed         = SEED,
)

print('Split distribution:')
print(combined['split'].value_counts())
print()
_ = summarize_split(
    combined,
    endpoint_col = 'endpoint_name',
    label_col    = 'label',
    split_col    = 'split',
)

In [ ]:
# ── Assign CV fold IDs within the train pool ──────────────────────────────────
train_df = combined[combined['split'] == 'train_pool'].copy()

train_df = make_cv_fold_assignments(
    train_df     = train_df,
    scaffold_col = 'scaffold',
    smiles_col   = 'canonical_smiles',
    endpoint_col = 'endpoint_name',
    label_col    = 'label',
    n_folds      = N_CV_FOLDS,
    seed         = SEED,
)

# Merge cv_fold back into combined
combined = combined.merge(
    train_df[['canonical_smiles', 'endpoint_name', 'cv_fold']],
    on  = ['canonical_smiles', 'endpoint_name'],
    how = 'left',
)

print('CV fold distribution (train pool only):')
print(train_df['cv_fold'].value_counts().sort_index())

---
## 9 — Leakage Check

**Critical validation:** No scaffold or canonical SMILES should appear in both  
the train pool and held-out test. If leakage is detected, the split function raises  
a `ValueError` and the notebook stops.

In [ ]:
leakage_summary = check_leakage(
    df           = combined,
    scaffold_col = 'scaffold',
    smiles_col   = 'canonical_smiles',
    split_col    = 'split',
)

print(pd.DataFrame([leakage_summary]).to_string(index=False))

---
## 10 — Save Outputs

Outputs are saved per endpoint in the format expected by downstream notebooks:

```
{split_root}/{endpoint_name}/train.xlsx   ← train pool
{split_root}/{endpoint_name}/test.xlsx    ← held-out test
{split_root}/combined_all_endpoints.csv  ← full multi-endpoint matrix
```

Each file contains columns: `Molecule ChEMBL ID`, `canonical_smiles`, `Smiles`, `pIC50`, `label`, `label_name`, `scaffold`, `cv_fold` (train only)

In [ ]:
OUTPUT_COLS = [
    'Molecule ChEMBL ID',
    'canonical_smiles',
    'Smiles',
    'pIC50',
    'label',
    'label_name',
    'scaffold',
    'endpoint_name',
    'split',
    'cv_fold',
]

# Keep only columns that actually exist in the DataFrame
OUTPUT_COLS = [c for c in OUTPUT_COLS if c in combined.columns]

for endpoint_name, grp in combined.groupby('endpoint_name'):
    endpoint_dir = SPLIT_ROOT / endpoint_name
    endpoint_dir.mkdir(parents=True, exist_ok=True)

    train = grp[grp['split'] == 'train_pool'][OUTPUT_COLS]
    test  = grp[grp['split'] == 'heldout_test'][OUTPUT_COLS]

    train.to_excel(endpoint_dir / 'train.xlsx', index=False)
    test.to_excel(endpoint_dir  / 'test.xlsx',  index=False)

    print(
        f'  [{endpoint_name}] '
        f'Train: {len(train):,} | Test: {len(test):,} | '
        f'Saved → {endpoint_dir}'
    )

# Save full combined matrix
combined_path = SPLIT_ROOT / 'combined_all_endpoints.csv'
combined[OUTPUT_COLS].to_csv(combined_path, index=False)
print(f'\n  Combined matrix saved → {combined_path}')
print('\n✅ All outputs saved.')

---
## 11 — Summary Statistics (Table 1)

This reproduces **Table 1** from the paper:  
*Dataset summary after curation and splitting.*

In [ ]:
rows = []
for endpoint_name, grp in combined.groupby('endpoint_name'):
    train = grp[grp['split'] == 'train_pool']
    test  = grp[grp['split'] == 'heldout_test']
    all_  = grp

    rows.append({
        'Endpoint':         endpoint_name,
        'Total (n)':        len(all_),
        'Active (n)':       (all_['label'] == 1).sum(),
        'Non-active (n)':   (all_['label'] == 0).sum(),
        'Ratio':            f"{(all_['label']==1).sum() / max((all_['label']==0).sum(),1):.2f}:1",
        'Unique scaffolds':  all_['scaffold'].nunique(),
        'Train pool (n)':   len(train),
        'Held-out test (n)': len(test),
    })

table1 = pd.DataFrame(rows).set_index('Endpoint')

# Add totals row
totals = {
    'Total (n)':          table1['Total (n)'].sum(),
    'Active (n)':         table1['Active (n)'].sum(),
    'Non-active (n)':     table1['Non-active (n)'].sum(),
    'Ratio':              '—',
    'Unique scaffolds':   combined['scaffold'].nunique(),
    'Train pool (n)':     table1['Train pool (n)'].sum(),
    'Held-out test (n)':  table1['Held-out test (n)'].sum(),
}
table1.loc['**TOTAL**'] = totals

print('=' * 80)
print('TABLE 1 — Dataset Summary After Curation and Splitting')
print('=' * 80)
print(table1.to_string())
print('=' * 80)

In [ ]:
# ── pIC50 distribution per endpoint ──────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['font.size'] = 11

    endpoints = combined['endpoint_name'].unique()
    n_cols = 3
    n_rows = int(np.ceil(len(endpoints) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
    axes = axes.flatten()

    colors = {'Active': '#E8593C', 'Non-active': '#3B8BD4'}

    for i, ep in enumerate(sorted(endpoints)):
        ax  = axes[i]
        grp = combined[combined['endpoint_name'] == ep]

        for label_name, color in colors.items():
            subset = grp[grp['label_name'] == label_name]['pIC50'].dropna()
            ax.hist(subset, bins=30, alpha=0.65, color=color,
                    label=label_name, edgecolor='white', linewidth=0.4)

        ax.axvline(6.0, color='gray', linestyle='--', linewidth=1,
                   label='pIC50 = 6 (1000 nM)')
        ax.axvline(5.0, color='gray', linestyle=':',  linewidth=1,
                   label='pIC50 = 5 (10000 nM)')

        ax.set_title(ep.replace('_IC50', ''), fontweight='bold')
        ax.set_xlabel('pIC50')
        ax.set_ylabel('Count')
        if i == 0:
            ax.legend(fontsize=9, framealpha=0.7)

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle('pIC50 Distribution per Endpoint', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(SPLIT_ROOT / 'pIC50_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved → {SPLIT_ROOT}/pIC50_distribution.png')

except ImportError:
    print('matplotlib not available — skipping plot.')

---
## ✅ Notebook 01 Complete

### What was produced

```
{split_root}/
├── AR_IC50/
│   ├── train.xlsx           ← train pool (80%)
│   └── test.xlsx            ← held-out test (20%)
├── CYP19A1_IC50/  ...
├── ESR1_alpha_IC50/ ...
├── ESR2_beta_IC50/ ...
├── HSD11B1_IC50/ ...
├── PPARG_IC50/ ...
├── combined_all_endpoints.csv
└── pIC50_distribution.png
```

### Next notebook

➡️ **[02_chemical_space_and_ad.ipynb](02_chemical_space_and_ad.ipynb)**  
Chemical space visualization (UMAP), scaffold diversity analysis, and Tanimoto similarity distribution between train and test.